In [ ]:
pip uninstall -y torch torchvision torchaudio

In [ ]:
pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
pip install ema-pytorch

In [ ]:
import math
import copy
from pathlib import Path
from random import random
from functools import partial
from collections import namedtuple
from multiprocessing import cpu_count

import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch.nn import Module, ModuleList
from torch.amp import autocast
from torch.utils.data import Dataset, DataLoader

from torch.optim import Adam,AdamW

from torchvision import transforms as T, utils

from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange

from scipy.optimize import linear_sum_assignment

from PIL import Image
from tqdm.auto import tqdm
from ema_pytorch import EMA

from accelerate import Accelerator


In [ ]:
from functools import wraps
from packaging import version
from collections import namedtuple

import torch
from torch import nn, einsum
import torch.nn.functional as F

from einops import rearrange

# constants
AttentionConfig = namedtuple('AttentionConfig', ['backends'])

# helpers
def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d

def once(fn):
    called = False
    @wraps(fn)
    def inner(x):
        nonlocal called
        if called:
            return
        called = True
        return fn(x)
    return inner

print_once = once(print)

# main class
class Attend(nn.Module):
    def __init__(
        self,
        dropout = 0.,
        flash = False,
        scale = None
    ):
        super().__init__()
        self.dropout = dropout
        self.scale = scale
        self.attn_dropout = nn.Dropout(dropout)

        self.flash = flash
        assert not (flash and version.parse(torch.__version__) < version.parse('2.0.0')), \
            'in order to use flash attention, you must be using pytorch 2.0 or above'

        # determine efficient attention configs for cuda and cpu
        self.cpu_config = None
        self.cuda_config = None

        # Try to import SDPBackend (available in PyTorch 2.0+)
        try:
            from torch.nn.attention import SDPBackend
            self.cpu_config = AttentionConfig([
                SDPBackend.FLASH_ATTENTION,
                SDPBackend.MATH,
                SDPBackend.EFFICIENT_ATTENTION
            ])
        except ImportError:
            # Fallback for older PyTorch versions
            self.flash = False

        if not torch.cuda.is_available() or not flash:
            return

        device_properties = torch.cuda.get_device_properties(torch.device('cuda'))
        device_version = version.parse(f'{device_properties.major}.{device_properties.minor}')

        if device_version > version.parse('8.0'):
            print_once('A100 GPU detected, using flash attention if input tensor is on cuda')
            try:
                from torch.nn.attention import SDPBackend
                self.cuda_config = AttentionConfig([SDPBackend.FLASH_ATTENTION])
            except ImportError:
                self.flash = False
        else:
            print_once('Non-A100 GPU detected, using math or mem efficient attention if input tensor is on cuda')
            try:
                from torch.nn.attention import SDPBackend
                self.cuda_config = AttentionConfig([SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION])
            except ImportError:
                self.flash = False

    def manual_attn(self, q, k, v):
        """Fallback manual attention implementation."""
        scale = default(self.scale, q.shape[-1] ** -0.5)
        sim = einsum(f"b h i d, b h j d -> b h i j", q, k) * scale
        attn = sim.softmax(dim=-1)
        attn = self.attn_dropout(attn)
        out = einsum(f"b h i j, b h j d -> b h i d", attn, v)
        return out

    def flash_attn(self, q, k, v):
        _, heads, q_len, _, k_len, is_cuda, device = *q.shape, k.shape[-2], q.is_cuda, q.device

        if exists(self.scale):
            default_scale = q.shape[-1]
            q = q * (self.scale / default_scale)

        q, k, v = map(lambda t: t.contiguous(), (q, k, v))

        # Determine config based on device
        config = self.cuda_config if is_cuda else self.cpu_config

        # If no config (e.g., older PyTorch), fallback to manual
        if config is None:
            return self.manual_attn(q, k, v)

        try:
            with torch.nn.attention.sdpa_kernel(**config._asdict()):
                out = F.scaled_dot_product_attention(
                    q, k, v,
                    dropout_p=self.dropout if self.training else 0.
                )
            return out
        except (RuntimeError, AttributeError) as e:
            # Fallback if flash attention fails (e.g., unsupported hardware/driver)
            print_once(f'Flash attention failed: {e}. Falling back to manual attention.')
            return self.manual_attn(q, k, v)

    def forward(self, q, k, v):
        """
        einstein notation
        b - batch
        h - heads
        n, i, j - sequence length (base sequence length, source, target)
        d - feature dimension
        """
        if self.flash:
            return self.flash_attn(q, k, v)
        else:
            return self.manual_attn(q, k, v)

In [ ]:
# Sửa ModelPrediction để có thêm trường pred_mask_logits
ModelPrediction = namedtuple('ModelPrediction', ['pred_noise', 'pred_x_start', 'pred_mask_logits'])

In [ ]:
def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d

def cast_tuple(t, length = 1):
    if isinstance(t, tuple):
        return t
    return ((t,) * length)

def divisible_by(numer, denom):
    return (numer % denom) == 0

def identity(t, *args, **kwargs):
    return t

def cycle(dl):
    while True:
        for data in dl:
            yield data

def has_int_squareroot(num):
    return (math.sqrt(num) ** 2) == num

def num_to_groups(num, divisor):
    groups = num // divisor
    remainder = num % divisor
    arr = [divisor] * groups
    if remainder > 0:
        arr.append(remainder)
    return arr

def convert_image_to_fn(img_type, image):
    if image.mode != img_type:
        return image.convert(img_type)
    return image

def normalize_to_neg_one_to_one(img):
    return img * 2 - 1

def unnormalize_to_zero_to_one(t):
    return (t + 1) * 0.5


In [ ]:
def Upsample(dim, dim_out = None):
    return nn.Sequential(
        nn.Upsample(scale_factor = 2, mode = 'nearest'),
        nn.Conv2d(dim, default(dim_out, dim), 3, padding = 1)
    )

def Downsample(dim, dim_out = None):
    return nn.Sequential(
        Rearrange('b c (h p1) (w p2) -> b (c p1 p2) h w', p1 = 2, p2 = 2),
        nn.Conv2d(dim * 4, default(dim_out, dim), 1)
    )

class RMSNorm(Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = dim ** 0.5
        self.g = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        return F.normalize(x, dim = 1) * self.g * self.scale

class SinusoidalPosEmb(Module):
    def __init__(self, dim, theta = 10000):
        super().__init__()
        self.dim = dim
        self.theta = theta

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(self.theta) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class RandomOrLearnedSinusoidalPosEmb(Module):
    def __init__(self, dim, is_random = False):
        super().__init__()
        assert divisible_by(dim, 2)
        half_dim = dim // 2
        self.weights = nn.Parameter(torch.randn(half_dim), requires_grad = not is_random)

    def forward(self, x):
        x = rearrange(x, 'b -> b 1')
        freqs = x * rearrange(self.weights, 'd -> 1 d') * 2 * math.pi
        fouriered = torch.cat((freqs.sin(), freqs.cos()), dim = -1)
        fouriered = torch.cat((x, fouriered), dim = -1)
        return fouriered

class Block(Module):
    def __init__(self, dim, dim_out, dropout = 0.):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding = 1)
        self.norm = RMSNorm(dim_out)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, scale_shift = None):
        x = self.proj(x)
        x = self.norm(x)

        if exists(scale_shift):
            scale, shift = scale_shift
            x = x * (scale + 1) + shift

        x = self.act(x)
        return self.dropout(x)

class ResnetBlock(Module):
    def __init__(self, dim, dim_out, *, time_emb_dim = None, dropout = 0.):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, dim_out * 2)
        ) if exists(time_emb_dim) else None

        self.block1 = Block(dim, dim_out, dropout = dropout)
        self.block2 = Block(dim_out, dim_out)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb = None):

        scale_shift = None
        if exists(self.mlp) and exists(time_emb):
            time_emb = self.mlp(time_emb)
            time_emb = rearrange(time_emb, 'b c -> b c 1 1')
            scale_shift = time_emb.chunk(2, dim = 1)

        h = self.block1(x, scale_shift = scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class LinearAttention(Module):
    def __init__(self, dim, heads = 4, dim_head = 32, num_mem_kv = 4):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads

        self.norm = RMSNorm(dim)

        self.mem_kv = nn.Parameter(torch.randn(2, heads, dim_head, num_mem_kv))
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)

        self.to_out = nn.Sequential(
            nn.Conv2d(hidden_dim, dim, 1),
            RMSNorm(dim)
        )

    def forward(self, x):
        b, c, h, w = x.shape
        x = self.norm(x)
        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h = self.heads), qkv)

        mk, mv = map(lambda t: repeat(t, 'h c n -> b h c n', b = b), self.mem_kv)
        k, v = map(partial(torch.cat, dim = -1), ((mk, k), (mv, v)))

        q = q.softmax(dim = -2)
        k = k.softmax(dim = -1)
        q = q * self.scale

        context = torch.einsum('b h d n, b h e n -> b h d e', k, v)
        out = torch.einsum('b h d e, b h d n -> b h e n', context, q)
        out = rearrange(out, 'b h c (x y) -> b (h c) x y', h = self.heads, x = h, y = w)
        return self.to_out(out)

class Attention(Module):
    def __init__(self, dim, heads = 4, dim_head = 32, num_mem_kv = 4, flash = False):
        super().__init__()
        self.heads = heads
        hidden_dim = dim_head * heads

        self.norm = RMSNorm(dim)
        self.attend = Attend(flash = flash)

        self.mem_kv = nn.Parameter(torch.randn(2, heads, num_mem_kv, dim_head))
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        x = self.norm(x)
        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h (x y) c', h = self.heads), qkv)

        mk, mv = map(lambda t: repeat(t, 'h n d -> b h n d', b = b), self.mem_kv)
        k, v = map(partial(torch.cat, dim = -2), ((mk, k), (mv, v)))

        out = self.attend(q, k, v)
        out = rearrange(out, 'b h (x y) d -> b (h d) x y', x = h, y = w)
        return self.to_out(out)


In [ ]:
class Unet(Module):
    def __init__(
        self,
        dim,
        init_dim = None,
        out_dim_image = None,
        out_dim_mask = None,
        dim_mults = (1, 2, 4, 8),
        channels = 3,
        num_classes = 2,
        self_condition = False,
        learned_variance = False,
        learned_sinusoidal_cond = False,
        random_fourier_features = False,
        learned_sinusoidal_dim = 16,
        sinusoidal_pos_emb_theta = 10000,
        dropout = 0.,
        attn_dim_head = 32,
        attn_heads = 4,
        full_attn = None,    # defaults to full attention only for inner most layer
        flash_attn = False
    ):
        super().__init__()

        # determine dimensions

        self.channels = channels
        self.num_classes = num_classes
        self.self_condition = self_condition

        # input: image (channels) concatenated with mask (num_classes)
        input_channels = channels + num_classes
        input_channels = input_channels * (2 if self_condition else 1)

        init_dim = default(init_dim, dim)
        self.init_conv = nn.Conv2d(input_channels, init_dim, 7, padding = 3)

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:]))

        # time embeddings

        time_dim = dim * 4

        self.random_or_learned_sinusoidal_cond = learned_sinusoidal_cond or random_fourier_features

        if self.random_or_learned_sinusoidal_cond:
            sinu_pos_emb = RandomOrLearnedSinusoidalPosEmb(learned_sinusoidal_dim, random_fourier_features)
            fourier_dim = learned_sinusoidal_dim + 1
        else:
            sinu_pos_emb = SinusoidalPosEmb(dim, theta = sinusoidal_pos_emb_theta)
            fourier_dim = dim

        self.time_mlp = nn.Sequential(
            sinu_pos_emb,
            nn.Linear(fourier_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        # attention

        if not full_attn:
            full_attn = (*((False,) * (len(dim_mults) - 1)), True)

        num_stages = len(dim_mults)
        full_attn  = cast_tuple(full_attn, num_stages)
        attn_heads = cast_tuple(attn_heads, num_stages)
        attn_dim_head = cast_tuple(attn_dim_head, num_stages)

        assert len(full_attn) == len(dim_mults)

        # prepare blocks

        FullAttention = partial(Attention, flash = flash_attn)
        resnet_block = partial(ResnetBlock, time_emb_dim = time_dim, dropout = dropout)

        # layers

        self.downs = ModuleList([])
        self.ups = ModuleList([])
        num_resolutions = len(in_out)

        for ind, ((dim_in, dim_out), layer_full_attn, layer_attn_heads, layer_attn_dim_head) in enumerate(zip(in_out, full_attn, attn_heads, attn_dim_head)):
            is_last = ind >= (num_resolutions - 1)

            attn_klass = FullAttention if layer_full_attn else LinearAttention

            self.downs.append(ModuleList([
                resnet_block(dim_in, dim_in),
                resnet_block(dim_in, dim_in),
                attn_klass(dim_in, dim_head = layer_attn_dim_head, heads = layer_attn_heads),
                Downsample(dim_in, dim_out) if not is_last else nn.Conv2d(dim_in, dim_out, 3, padding = 1)
            ]))

        mid_dim = dims[-1]
        self.mid_block1 = resnet_block(mid_dim, mid_dim)
        self.mid_attn = FullAttention(mid_dim, heads = attn_heads[-1], dim_head = attn_dim_head[-1])
        self.mid_block2 = resnet_block(mid_dim, mid_dim)

        for ind, ((dim_in, dim_out), layer_full_attn, layer_attn_heads, layer_attn_dim_head) in enumerate(zip(*map(reversed, (in_out, full_attn, attn_heads, attn_dim_head)))):
            is_last = ind == (len(in_out) - 1)

            attn_klass = FullAttention if layer_full_attn else LinearAttention

            self.ups.append(ModuleList([
                resnet_block(dim_out + dim_in, dim_out),
                resnet_block(dim_out + dim_in, dim_out),
                attn_klass(dim_out, dim_head = layer_attn_dim_head, heads = layer_attn_heads),
                Upsample(dim_out, dim_in) if not is_last else nn.Conv2d(dim_out, dim_in, 3, padding = 1)
            ]))

        # two separate output heads: image and mask

        default_out_dim_image = channels * (1 if not learned_variance else 2)
        self.out_dim_image = default(out_dim_image, default_out_dim_image)
        self.out_dim_mask = default(out_dim_mask, num_classes)

        self.final_res_block = resnet_block(init_dim * 2, init_dim)
        self.final_conv_image = nn.Conv2d(init_dim, self.out_dim_image, 1)
        self.final_conv_mask = nn.Conv2d(init_dim, self.out_dim_mask, 1)

    @property
    def downsample_factor(self):
        return 2 ** (len(self.downs) - 1)

    def forward(self, x_img, x_mask, time, x_self_cond_img = None, x_self_cond_mask = None):
        assert all([divisible_by(d, self.downsample_factor) for d in x_img.shape[-2:]]), \
            f'your input dimensions {x_img.shape[-2:]} need to be divisible by {self.downsample_factor}, given the unet'

        # concat image and mask along channel dim
        x = torch.cat((x_img, x_mask), dim = 1)

        if self.self_condition:
            x_self_cond_img = default(x_self_cond_img, lambda: torch.zeros_like(x_img))
            x_self_cond_mask = default(x_self_cond_mask, lambda: torch.zeros_like(x_mask))
            x_self_cond = torch.cat((x_self_cond_img, x_self_cond_mask), dim = 1)
            x = torch.cat((x_self_cond, x), dim = 1)

        x = self.init_conv(x)
        r = x.clone()

        t = self.time_mlp(time)

        h = []

        for block1, block2, attn, downsample in self.downs:
            x = block1(x, t)
            h.append(x)

            x = block2(x, t)
            x = attn(x) + x
            h.append(x)

            x = downsample(x)

        x = self.mid_block1(x, t)
        x = self.mid_attn(x) + x
        x = self.mid_block2(x, t)

        for block1, block2, attn, upsample in self.ups:
            x = torch.cat((x, h.pop()), dim = 1)
            x = block1(x, t)

            x = torch.cat((x, h.pop()), dim = 1)
            x = block2(x, t)
            x = attn(x) + x

            x = upsample(x)

        x = torch.cat((x, r), dim = 1)

        x = self.final_res_block(x, t)

        out_img = self.final_conv_image(x)
        out_mask = self.final_conv_mask(x)

        return out_img, out_mask

In [ ]:
# Gaussian Diffusion functions 
# ============================================================

def extract(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def linear_beta_schedule(timesteps):
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype = torch.float64)

def cosine_beta_schedule(timesteps, s = 0.008):
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype = torch.float64) / timesteps
    alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

def sigmoid_beta_schedule(timesteps, start = -3, end = 3, tau = 1, clamp_min = 1e-5):
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype = torch.float64) / timesteps
    v_start = torch.tensor(start / tau).sigmoid()
    v_end = torch.tensor(end / tau).sigmoid()
    alphas_cumprod = (-((t * (end - start) + start) / tau).sigmoid() + v_end) / (v_end - v_start)
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from einops import rearrange, reduce
from tqdm import tqdm
from functools import partial
from collections import namedtuple



class MultiTaskDiffusion(nn.Module):
    def __init__(
        self,
        model,
        *,
        image_size=128,
        num_classes,
        timesteps=1000,
        sampling_timesteps=None,
        objective='pred_x0',
        beta_schedule='sigmoid',
        schedule_fn_kwargs=dict(),
        ddim_sampling_eta=0.,
        auto_normalize=True,
        offset_noise_strength=0.,
        min_snr_loss_weight=False,
        min_snr_gamma=5,
        balancing_factor=0.684,
        immiscible=False
    ):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.channels = model.channels

        if isinstance(image_size, int):
            image_size = (image_size, image_size)
        assert isinstance(image_size, (tuple, list)) and len(image_size) == 2
        self.image_size = image_size

        self.objective = objective
        assert objective in {'pred_noise', 'pred_x0', 'pred_v'}, 'objective must be pred_noise, pred_x0, or pred_v'

        # Beta schedule
        if beta_schedule == 'linear':
            beta_schedule_fn = linear_beta_schedule
        elif beta_schedule == 'cosine':
            beta_schedule_fn = cosine_beta_schedule
        elif beta_schedule == 'sigmoid':
            beta_schedule_fn = sigmoid_beta_schedule
        else:
            raise ValueError(f'unknown beta schedule {beta_schedule}')

        betas = beta_schedule_fn(timesteps, **schedule_fn_kwargs)

        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.)

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)

        self.sampling_timesteps = default(sampling_timesteps, timesteps)
        assert self.sampling_timesteps <= timesteps
        self.is_ddim_sampling = self.sampling_timesteps < timesteps
        self.ddim_sampling_eta = ddim_sampling_eta

        register_buffer = lambda name, val: self.register_buffer(name, val.to(torch.float32))

        register_buffer('betas', betas)
        register_buffer('alphas_cumprod', alphas_cumprod)
        register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)

        register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1. - alphas_cumprod))
        register_buffer('log_one_minus_alphas_cumprod', torch.log(1. - alphas_cumprod))
        register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1. / alphas_cumprod))
        register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1. / alphas_cumprod - 1))

        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        register_buffer('posterior_variance', posterior_variance)
        register_buffer('posterior_log_variance_clipped', torch.log(posterior_variance.clamp(min=1e-20)))
        register_buffer('posterior_mean_coef1', betas * torch.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))
        register_buffer('posterior_mean_coef2', (1. - alphas_cumprod_prev) * torch.sqrt(alphas) / (1. - alphas_cumprod))

        self.immiscible = immiscible
        self.offset_noise_strength = offset_noise_strength

        snr = alphas_cumprod / (1 - alphas_cumprod)
        maybe_clipped_snr = snr.clone()
        if min_snr_loss_weight:
            maybe_clipped_snr.clamp_(max=min_snr_gamma)

        if objective == 'pred_noise':
            register_buffer('loss_weight_image', maybe_clipped_snr / snr)
        elif objective == 'pred_x0':
            register_buffer('loss_weight_image', maybe_clipped_snr)
        elif objective == 'pred_v':
            register_buffer('loss_weight_image', maybe_clipped_snr / (snr + 1))

        if balancing_factor == 'snr':
            balancing_factor_values = torch.sqrt(snr)
        elif isinstance(balancing_factor, (int, float)):
            balancing_factor_values = torch.full_like(snr, float(balancing_factor))
        else:
            raise ValueError(f"balancing_factor must be 'snr' or a number, got {balancing_factor}")
        register_buffer('balancing_factor', balancing_factor_values)

        # Tự động chuẩn hóa ảnh về [-1, 1] nếu auto_normalize = True
        self.auto_normalize = auto_normalize
        self.normalize = normalize_to_neg_one_to_one if auto_normalize else identity
        self.unnormalize = unnormalize_to_zero_to_one if auto_normalize else identity

    @property
    def device(self):
        return self.betas.device

    # ---------- Các hàm dự đoán cơ bản ----------
    def predict_start_from_noise(self, x_t, t, noise):
        return (
            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
        )

    def predict_noise_from_start(self, x_t, t, x0):
        return (
            (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0) /
            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)
        )

    def predict_v(self, x_start, t, noise):
        return (
            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * noise -
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * x_start
        )

    def predict_start_from_v(self, x_t, t, v):
        return (
            extract(self.sqrt_alphas_cumprod, t, x_t.shape) * x_t -
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * v
        )

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = (
            extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
            extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        posterior_variance = extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped

    def model_predictions(self, x_img, x_mask, t, x_self_cond_img=None, x_self_cond_mask=None,
                          clip_x_start=False, rederive_pred_noise=False):
        model_out_img, model_out_mask = self.model(x_img, x_mask, t, x_self_cond_img, x_self_cond_mask)
        maybe_clip = partial(torch.clamp, min=-1., max=1.) if clip_x_start else identity

        if self.objective == 'pred_noise':
            pred_noise = model_out_img
            x_start = self.predict_start_from_noise(x_img, t, pred_noise)
            x_start = maybe_clip(x_start)
            if clip_x_start and rederive_pred_noise:
                pred_noise = self.predict_noise_from_start(x_img, t, x_start)
        elif self.objective == 'pred_x0':
            x_start = model_out_img
            x_start = maybe_clip(x_start)
            pred_noise = self.predict_noise_from_start(x_img, t, x_start)
        elif self.objective == 'pred_v':
            v = model_out_img
            x_start = self.predict_start_from_v(x_img, t, v)
            x_start = maybe_clip(x_start)
            pred_noise = self.predict_noise_from_start(x_img, t, x_start)
        else:
            raise ValueError(f'unknown objective {self.objective}')

        return ModelPrediction(pred_noise, x_start, model_out_mask)

    # ---------- Training Loss ----------
    def p_losses(self, x_start_img, x_start_mask_onehot, t, noise=None):
        # 1. Normalize và scale mask
        y0 = x_start_mask_onehot * 2 - 1
        b_factor = extract(self.balancing_factor, t, y0.shape)
        y0_prime = y0 * b_factor

        # 2. Ghép ảnh và mask
        x_combined = torch.cat([x_start_img, y0_prime], dim=1)

        # 3. Tạo noise chung
        if noise is None:
            noise_combined = torch.randn_like(x_combined)
        else:
            noise_combined = noise

        # Offset noise cho phần ảnh 
        if self.offset_noise_strength > 0.:
            offset_noise = torch.randn(x_start_img.shape[:2], device=self.device)
            noise_img_part = noise_combined[:, :self.channels] + self.offset_noise_strength * rearrange(offset_noise, 'b c -> b c 1 1')
            noise_combined = torch.cat([noise_img_part, noise_combined[:, self.channels:]], dim=1)

        # 4. Thêm nhiễu
        sqrt_alpha = extract(self.sqrt_alphas_cumprod, t, x_combined.shape)
        sqrt_one_minus = extract(self.sqrt_one_minus_alphas_cumprod, t, x_combined.shape)
        x_t_combined = sqrt_alpha * x_combined + sqrt_one_minus * noise_combined

        # 5. Tách ra
        x_t_img = x_t_combined[:, :self.channels]
        x_t_mask = x_t_combined[:, self.channels:]  # y'_t

        # 6. Dự đoán
        model_out_img, model_out_mask = self.model(x_t_img, x_t_mask, t)

        # 7. Loss ảnh
        if self.objective == 'pred_noise':
            target_img = noise_combined[:, :self.channels]
        elif self.objective == 'pred_x0':
            target_img = x_start_img
        elif self.objective == 'pred_v':
            target_img = self.predict_v(x_start_img, t, noise_combined[:, :self.channels])
        else:
            raise ValueError(f'unknown objective {self.objective}')

        loss_img = F.mse_loss(model_out_img, target_img, reduction='none')
        loss_img = reduce(loss_img, 'b ... -> b', 'mean')
        loss_img = loss_img * extract(self.loss_weight_image, t, loss_img.shape)
        loss_img = loss_img.mean()

        # 8. Loss mask
        target_mask = x_start_mask_onehot.argmax(dim=1)
        loss_mask = F.cross_entropy(model_out_mask, target_mask, reduction='mean')

        return loss_img + loss_mask

    def forward(self, img, mask_onehot, *args, **kwargs):
        # Chuẩn hóa ảnh về [-1, 1] nếu auto_normalize = True
        img = self.normalize(img)

        b, c, h, w = img.shape
        assert h == self.image_size[0] and w == self.image_size[1], \
            f"Image size {h}x{w} không khớp với {self.image_size}"
        t = torch.randint(0, self.num_timesteps, (b,), device=img.device).long()
        return self.p_losses(img, mask_onehot, t, *args, **kwargs)

    # ---------- DDPM Sampling ----------
    @torch.inference_mode()
    def p_sample_loop(self, batch_size, return_all_timesteps=False):
        device = self.device
        shape_img = (batch_size, self.channels, *self.image_size)
        shape_mask = (batch_size, self.num_classes, *self.image_size)

        img = torch.randn(shape_img, device=device)   # x_T
        mask = torch.randn(shape_mask, device=device) # y'_T
        imgs = [img]

        x_start = None
        mask_logits = None

        for t in tqdm(reversed(range(0, self.num_timesteps)), desc='DDPM sampling', total=self.num_timesteps):
            batched_times = torch.full((batch_size,), t, device=device, dtype=torch.long)

            # 1. Dự đoán với img và mask hiện tại (cùng ở timestep t)
            preds = self.model_predictions(img, mask, batched_times,
                                           x_self_cond_img=x_start,
                                           x_self_cond_mask=mask_logits,
                                           clip_x_start=True, rederive_pred_noise=True)
            x_start = preds.pred_x_start
            pred_noise = preds.pred_noise
            mask_logits = preds.pred_mask_logits

            # 2. Rescale mask logits
            probs = F.softmax(mask_logits, dim=1)
            b_factor = extract(self.balancing_factor, batched_times, probs.shape)
            y_hat_scaled = (probs * 2 - 1) * b_factor

            # 3. Cập nhật mask bằng posterior (DDPM step)
            if t > 0:
                alpha = self.alphas_cumprod[t]
                alpha_prev = self.alphas_cumprod_prev[t]
                posterior_variance = extract(self.posterior_variance, batched_times, mask.shape)
                coef1 = extract(self.posterior_mean_coef1, batched_times, y_hat_scaled.shape)
                coef2 = extract(self.posterior_mean_coef2, batched_times, y_hat_scaled.shape)
                model_mean_mask = coef1 * y_hat_scaled + coef2 * mask
                noise_mask = torch.randn_like(mask)
                mask = model_mean_mask + (0.5 * torch.log(posterior_variance)).exp() * noise_mask
            else:
                mask = y_hat_scaled

            # 4. Cập nhật ảnh bằng posterior (DDPM step)
            if t > 0:
                posterior_mean_img, posterior_variance_img, posterior_log_variance_img = self.q_posterior(
                    x_start=x_start, x_t=img, t=batched_times
                )
                noise_img = torch.randn_like(img)
                img = posterior_mean_img + (0.5 * posterior_log_variance_img).exp() * noise_img
            else:
                img = x_start

            imgs.append(img)

        ret_img = img if not return_all_timesteps else torch.stack(imgs, dim=1)
        ret_img = self.unnormalize(ret_img)
        return ret_img, mask_logits

    # ---------- DDIM Sampling ----------
    @torch.inference_mode()
    def ddim_sample(self, batch_size=16, return_all_timesteps=False):
        device = self.device
        total_timesteps = self.num_timesteps
        sampling_timesteps = self.sampling_timesteps
        eta = self.ddim_sampling_eta

        times = torch.linspace(-1, total_timesteps - 1, steps=sampling_timesteps + 1)
        times = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))

        shape_img = (batch_size, self.channels, *self.image_size)
        shape_mask = (batch_size, self.num_classes, *self.image_size)

        img = torch.randn(shape_img, device=device)
        mask = torch.randn(shape_mask, device=device)

        imgs = [img]
        x_start = None
        mask_logits = None

        for time, time_next in tqdm(time_pairs, desc='DDIM sampling'):
            time_cond = torch.full((batch_size,), time, device=device, dtype=torch.long)

            preds = self.model_predictions(img, mask, time_cond,
                                          x_self_cond_img=x_start,
                                          x_self_cond_mask=mask_logits,
                                          clip_x_start=True, rederive_pred_noise=True)
            x_start = preds.pred_x_start
            mask_logits = preds.pred_mask_logits

            if time_next < 0:
                img = x_start
                imgs.append(img)
                break

            alpha = self.alphas_cumprod[time]
            alpha_next = self.alphas_cumprod[time_next]

            probs = F.softmax(mask_logits, dim=1)
            y_hat_scaled = (probs * 2 - 1) * extract(self.balancing_factor, time_cond, probs.shape)

            sigma = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
            c = (1 - alpha_next - sigma ** 2).sqrt()

            noise_img = torch.randn_like(img)
            noise_mask = torch.randn_like(mask)

            img = x_start * alpha_next.sqrt() + c * preds.pred_noise + sigma * noise_img

            noise_mask_pred = (mask - alpha.sqrt() * y_hat_scaled) / (1 - alpha).sqrt()
            mask = y_hat_scaled * alpha_next.sqrt() + c * noise_mask_pred + sigma * noise_mask

            imgs.append(img)

        img = self.unnormalize(img)
        if return_all_timesteps:
            imgs = torch.stack(imgs, dim=1)
            imgs = self.unnormalize(imgs)
            return imgs, mask_logits
        return img, mask_logits

    def sample(self, batch_size=16, return_all_timesteps=False):
        if self.is_ddim_sampling:
            return self.ddim_sample(batch_size, return_all_timesteps)
        else:
            return self.p_sample_loop(batch_size, return_all_timesteps)

In [ ]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.transforms as T

# ===== helper =====
def exists(x):
    return x is not None

def identity(x):
    return x

# ===== dataset =====
class ImageMaskDataset(Dataset):
    def __init__(
        self,
        folder,
        image_size,
        num_classes=2,
        split='train',
        augment_horizontal_flip=True,
        convert_image_to='RGB'
    ):
        super().__init__()

        # ===== FIX SIZE =====
        if isinstance(image_size, int):
            self.image_size = (image_size, image_size)
        elif isinstance(image_size, (list, tuple)) and len(image_size) == 2:
            self.image_size = tuple(image_size)
        else:
            raise ValueError("image_size phải là int hoặc tuple (H, W)")

        self.folder = Path(folder) / split
        self.num_classes = num_classes

        img_dir = self.folder / 'images'
        mask_dir = self.folder / 'masks'

        self.img_paths = sorted(list(img_dir.glob('*')))
        self.mask_paths = sorted(list(mask_dir.glob('*')))

        self.img_dict = {p.stem: p for p in self.img_paths}
        self.mask_dict = {p.stem: p for p in self.mask_paths}

        self.stems = sorted(list(set(self.img_dict) & set(self.mask_dict)))

        # ===== TRANSFORMS =====
        self.transform_img = T.Compose([
            T.Resize(self.image_size),
            T.RandomHorizontalFlip() if augment_horizontal_flip else T.Lambda(lambda x: x),
            T.ToTensor(),   # (3, H, W)
        ])

        self.transform_mask = T.Compose([
            T.Resize(self.image_size, interpolation=T.InterpolationMode.NEAREST),
            T.RandomHorizontalFlip() if augment_horizontal_flip else T.Lambda(lambda x: x),
            T.ToTensor(),   # (1, H, W)
        ])

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, index):
        stem = self.stems[index]

        img_path = self.img_dict[stem]
        mask_path = self.mask_dict[stem]

        # ===== LOAD =====
        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')  # grayscale

        # ===== SYNC AUGMENT =====
        seed = torch.randint(0, 2**32, (1,)).item()

        torch.manual_seed(seed)
        img = self.transform_img(img)

        torch.manual_seed(seed)
        mask = self.transform_mask(mask)

        # ===== FIX MASK =====
        # (1, H, W) -> (H, W)
        if mask.dim() == 3:
            mask = mask.squeeze(0)

        # binary hóa
        mask = (mask > 0).long()   # (H, W)

        # ===== ONE HOT =====
        mask_onehot = F.one_hot(mask, num_classes=self.num_classes)  # (H, W, C)

        # đảm bảo đúng dim trước permute
        if mask_onehot.dim() == 3:
            mask_onehot = mask_onehot.permute(2, 0, 1).float()  # (C, H, W)
        else:
            raise ValueError(f"Mask bị sai shape: {mask_onehot.shape}")

        return img, mask_onehot

In [ ]:
class Trainer:
  def __init__(
    self,
    diffusion_model,
    folder,
    *,
    train_batch_size=16,
    gradient_accumulate_every=1,
    augment_horizontal_flip=True,
    train_lr=2e-5,
    train_num_steps=100000,
    ema_update_every=10,
    ema_decay=0.995,
    adam_betas=(0.9, 0.99),
    save_and_sample_every=1000,
    num_samples=10,
    results_folder='/kaggle/working/results',
    amp=False,
    mixed_precision_type='fp16',
    split_batches=True,
    convert_image_to=None,
    calculate_fid=False,                # tắt FID mặc định vì cần tùy chỉnh cho ảnh SAR
    max_grad_norm=1.,
    num_classes=2,                      # số lớp phân đoạn
    save_best_and_latest_only=False
):
    super().__init__()

    # accelerator
    self.accelerator = Accelerator(
        split_batches=split_batches,
        mixed_precision=mixed_precision_type if amp else 'no'
    )

    self.model = diffusion_model
    self.channels = diffusion_model.channels
    self.num_classes = num_classes
    is_ddim_sampling = diffusion_model.is_ddim_sampling

    # default convert_image_to
    if not exists(convert_image_to):
        convert_image_to = {1: 'L', 3: 'RGB', 4: 'RGBA'}.get(self.channels)

    # sampling and training hyperparameters
    assert has_int_squareroot(num_samples), 'number of samples must have an integer square root'
    self.num_samples = num_samples
    self.save_and_sample_every = save_and_sample_every

    self.batch_size = train_batch_size
    self.gradient_accumulate_every = gradient_accumulate_every
    assert (train_batch_size * gradient_accumulate_every) >= 16, \
        f'effective batch size should be at least 16'

    self.train_num_steps = train_num_steps
    self.image_size = diffusion_model.image_size
    self.max_grad_norm = max_grad_norm

    # dataset and dataloader (sử dụng SARImageMaskDataset)
    self.ds = ImageMaskDataset(
        folder=folder,
        image_size=self.image_size,
        num_classes=num_classes,
        split='train',
        augment_horizontal_flip=augment_horizontal_flip,
        convert_image_to=convert_image_to
    )

    assert len(self.ds) >= 100

    dl = DataLoader(
        self.ds,
        batch_size=train_batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=cpu_count()
    )

    dl = self.accelerator.prepare(dl)
    self.dl = cycle(dl)

    # optimizer
    self.opt = AdamW(diffusion_model.parameters(), lr=train_lr, betas=adam_betas)

    # EMA
    if self.accelerator.is_main_process:
        self.ema = EMA(diffusion_model, beta=ema_decay, update_every=ema_update_every)
        self.ema.to(self.device)

    self.results_folder = Path(results_folder)
    self.results_folder.mkdir(exist_ok=True)

    self.step = 0

    # prepare model, optimizer with accelerator
    self.model, self.opt = self.accelerator.prepare(self.model, self.opt)

    # FID (tạm tắt vì cần tùy chỉnh cho dataset SAR)
    self.calculate_fid = calculate_fid and self.accelerator.is_main_process
    if self.calculate_fid:
        from denoising_diffusion_pytorch.fid_evaluation import FIDEvaluation
        if not is_ddim_sampling:
            self.accelerator.print(
                "WARNING: Robust FID computation requires many samples. Consider using DDIM sampling."
            )
        # Lưu ý: FIDEvaluation hiện tại chỉ hỗ trợ ảnh đơn kênh/3 kênh, có thể cần sửa để bỏ qua mask
        self.fid_scorer = FIDEvaluation(
            batch_size=self.batch_size,
            dl=self.dl,
            sampler=self.ema.ema_model,
            channels=self.channels,
            accelerator=self.accelerator,
            stats_dir=results_folder,
            device=self.device,
            num_fid_samples=50000,
            inception_block_idx=2048
        )

    self.save_best_and_latest_only = save_best_and_latest_only
    if save_best_and_latest_only:
        assert calculate_fid, "FID required for best model selection."
        self.best_fid = 1e10

@property
def device(self):
    return self.accelerator.device

def save(self, milestone):
    if not self.accelerator.is_local_main_process:
        return
    data = {
        'step': self.step,
        'model': self.accelerator.get_state_dict(self.model),
        'opt': self.opt.state_dict(),
        'ema': self.ema.state_dict(),
        'scaler': self.accelerator.scaler.state_dict() if exists(self.accelerator.scaler) else None,
    }
    # Thêm version nếu có
    try:
        data['version'] = __version__
    except ImportError:
        pass
    torch.save(data, str(self.results_folder / f'model-{milestone}.pt'))

def load(self, milestone):
    accelerator = self.accelerator
    device = accelerator.device
    data = torch.load(
        str(self.results_folder / f'model-{milestone}.pt'),
        map_location=device,
        weights_only=True
    )
    model = self.accelerator.unwrap_model(self.model)
    model.load_state_dict(data['model'])
    self.step = data['step']
    self.opt.load_state_dict(data['opt'])
    if self.accelerator.is_main_process:
        self.ema.load_state_dict(data["ema"])
    if exists(self.accelerator.scaler) and exists(data['scaler']):
        self.accelerator.scaler.load_state_dict(data['scaler'])
    if 'version' in data:
        print(f"loading from version {data['version']}")

def train(self):
    accelerator = self.accelerator
    device = accelerator.device

    with tqdm(
        initial=self.step,
        total=self.train_num_steps,
        disable=not accelerator.is_main_process
    ) as pbar:

        while self.step < self.train_num_steps:
            self.model.train()
            total_loss = 0.

            for _ in range(self.gradient_accumulate_every):
                img, mask_onehot = next(self.dl)
                img = img.to(device)
                mask_onehot = mask_onehot.to(device)

                with self.accelerator.autocast():
                    loss = self.model(img, mask_onehot)
                    loss = loss / self.gradient_accumulate_every
                    total_loss += loss.item()

                self.accelerator.backward(loss)

            pbar.set_description(f'loss: {total_loss:.4f}')

            accelerator.wait_for_everyone()
            accelerator.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

            self.opt.step()
            self.opt.zero_grad()

            accelerator.wait_for_everyone()

            self.step += 1
            if accelerator.is_main_process:
                self.ema.update()

                if self.step != 0 and divisible_by(self.step, self.save_and_sample_every):
                    self.ema.ema_model.eval()

                    with torch.inference_mode():
                        milestone = self.step // self.save_and_sample_every
                        batches = num_to_groups(self.num_samples, self.batch_size)
                        all_images = []
                        all_masks = []

                        for n in batches:
                            # sample trả về (img, logits)
                            img, logits = self.ema.ema_model.sample(batch_size=n)
                            all_images.append(img)
                            # Tạo mask dự đoán (argmax)
                            mask_pred = logits.argmax(dim=1, keepdim=True).float() / (self.num_classes - 1)
                            all_masks.append(mask_pred)

                    all_images = torch.cat(all_images, dim=0)
                    all_masks = torch.cat(all_masks, dim=0)

                    # Lưu ảnh và mask
                    utils.save_image(
                        all_images,
                        str(self.results_folder / f'sample-{milestone}.png'),
                        nrow=int(math.sqrt(self.num_samples))
                    )
                    utils.save_image(
                        all_masks,
                        str(self.results_folder / f'sample-mask-{milestone}.png'),
                        nrow=int(math.sqrt(self.num_samples))
                    )

                    # FID (nếu bật)
                    if self.calculate_fid:
                        fid_score = self.fid_scorer.fid_score()
                        accelerator.print(f'fid_score: {fid_score}')
                        if self.save_best_and_latest_only:
                            if self.best_fid > fid_score:
                                self.best_fid = fid_score
                                self.save("best")
                            self.save("latest")
                    else:
                        self.save(milestone)

            pbar.update(1)

    accelerator.print('training complete')

In [ ]:
class Trainer:
    def __init__(
        self,
        diffusion_model,
        folder,
        *,
        train_batch_size=16,
        gradient_accumulate_every=1,
        augment_horizontal_flip=True,
        train_lr=2e-5,
        train_num_steps=100000,
        ema_update_every=10,
        ema_decay=0.995,
        adam_betas=(0.9, 0.99),
        save_and_sample_every=1000,
        num_samples=10,
        results_folder='/kaggle/working/results',
        amp=False,
        mixed_precision_type='fp16',
        split_batches=True,
        convert_image_to=None,
        calculate_fid=False,
        max_grad_norm=1.,
        num_classes=2,
        save_best_and_latest_only=False
    ):
        super().__init__()

        # accelerator
        self.accelerator = Accelerator(
            split_batches=split_batches,
            mixed_precision=mixed_precision_type if amp else 'no'
        )

        self.model = diffusion_model
        self.channels = diffusion_model.channels
        self.num_classes = num_classes
        is_ddim_sampling = diffusion_model.is_ddim_sampling

        # default convert_image_to
        if not exists(convert_image_to):
            convert_image_to = {1: 'L', 3: 'RGB', 4: 'RGBA'}.get(self.channels)

        # sampling and training hyperparameters
        assert has_int_squareroot(num_samples), 'number of samples must have an integer square root'
        self.num_samples = num_samples
        self.save_and_sample_every = save_and_sample_every

        self.batch_size = train_batch_size
        self.gradient_accumulate_every = gradient_accumulate_every
        assert (train_batch_size * gradient_accumulate_every) >= 16, \
            f'effective batch size should be at least 16'

        self.train_num_steps = train_num_steps
        self.image_size = diffusion_model.image_size
        self.max_grad_norm = max_grad_norm

        # dataset and dataloader
        self.ds = ImageMaskDataset(   # giả sử bạn đã định nghĩa lớp này
            folder=folder,
            image_size=self.image_size,
            num_classes=num_classes,
            split='train',
            augment_horizontal_flip=augment_horizontal_flip,
            convert_image_to=convert_image_to
        )

        assert len(self.ds) >= 100

        dl = DataLoader(
            self.ds,
            batch_size=train_batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=cpu_count()
        )

        dl = self.accelerator.prepare(dl)
        self.dl = cycle(dl)

        # optimizer
        self.opt = AdamW(diffusion_model.parameters(), lr=train_lr, betas=adam_betas)

        # EMA – sửa lỗi: dùng self.accelerator.device
        if self.accelerator.is_main_process:
            self.ema = EMA(diffusion_model, beta=ema_decay, update_every=ema_update_every)
            self.ema.to(self.accelerator.device)  

        self.results_folder = Path(results_folder)
        self.results_folder.mkdir(exist_ok=True)

        self.step = 0

        # prepare model, optimizer with accelerator
        self.model, self.opt = self.accelerator.prepare(self.model, self.opt)

        # FID – sửa lỗi: dùng self.accelerator.device
        self.calculate_fid = calculate_fid and self.accelerator.is_main_process
        if self.calculate_fid:
            if not is_ddim_sampling:
                self.accelerator.print(
                    "WARNING: Robust FID computation requires many samples. Consider using DDIM sampling."
                )
            self.fid_scorer = FIDEvaluation(
                batch_size=self.batch_size,
                dl=self.dl,
                sampler=self.ema.ema_model,
                channels=self.channels,
                accelerator=self.accelerator,
                stats_dir=results_folder,
                device=self.accelerator.device,  
                num_fid_samples=50000,
                inception_block_idx=2048
            )

        self.save_best_and_latest_only = save_best_and_latest_only
        if save_best_and_latest_only:
            assert calculate_fid, "FID required for best model selection."
            self.best_fid = 1e10

    # --- property device (đặt sau __init__) ---
    @property
    def device(self):
        return self.accelerator.device

    # --- các phương thức save, load, train ---
    def save(self, milestone):
        if not self.accelerator.is_local_main_process:
            return
        data = {
            'step': self.step,
            'model': self.accelerator.get_state_dict(self.model),
            'opt': self.opt.state_dict(),
            'ema': self.ema.state_dict(),
            'scaler': self.accelerator.scaler.state_dict() if exists(self.accelerator.scaler) else None,
        }
        try:
            from denoising_diffusion_pytorch import __version__
            data['version'] = __version__
        except ImportError:
            pass
        torch.save(data, str(self.results_folder / f'model-{milestone}.pt'))

    def load(self, milestone):
        accelerator = self.accelerator
        device = accelerator.device
        data = torch.load(
            str(self.results_folder / f'model-{milestone}.pt'),
            map_location=device,
            weights_only=True
        )
        model = self.accelerator.unwrap_model(self.model)
        model.load_state_dict(data['model'])
        self.step = data['step']
        self.opt.load_state_dict(data['opt'])
        if self.accelerator.is_main_process:
            self.ema.load_state_dict(data["ema"])
        if exists(self.accelerator.scaler) and exists(data['scaler']):
            self.accelerator.scaler.load_state_dict(data['scaler'])
        if 'version' in data:
            print(f"loading from version {data['version']}")

    def train(self):
        accelerator = self.accelerator
        device = accelerator.device

        with tqdm(
            initial=self.step,
            total=self.train_num_steps,
            disable=not accelerator.is_main_process
        ) as pbar:

            while self.step < self.train_num_steps:
                self.model.train()
                total_loss = 0.

                for _ in range(self.gradient_accumulate_every):
                    img, mask_onehot = next(self.dl)
                    img = img.to(device)
                    mask_onehot = mask_onehot.to(device)

                    with self.accelerator.autocast():
                        loss = self.model(img, mask_onehot)
                        loss = loss / self.gradient_accumulate_every
                        total_loss += loss.item()

                    self.accelerator.backward(loss)

                pbar.set_description(f'loss: {total_loss:.4f}')

                accelerator.wait_for_everyone()
                accelerator.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

                self.opt.step()
                self.opt.zero_grad()

                accelerator.wait_for_everyone()

                self.step += 1
                if accelerator.is_main_process:
                    self.ema.update()

                    if self.step != 0 and divisible_by(self.step, self.save_and_sample_every):
                        self.ema.ema_model.eval()

                        with torch.inference_mode():
                            milestone = self.step // self.save_and_sample_every
                            batches = num_to_groups(self.num_samples, self.batch_size)
                            all_images = []
                            all_masks = []

                            for n in batches:
                                # sample trả về (img, logits)
                                img, logits = self.ema.ema_model.sample(batch_size=n)
                                all_images.append(img)
                                # Tạo mask dự đoán (argmax)
                                mask_pred = logits.argmax(dim=1, keepdim=True).float() / (self.num_classes - 1)
                                all_masks.append(mask_pred)

                        all_images = torch.cat(all_images, dim=0)
                        all_masks = torch.cat(all_masks, dim=0)

                        # Lưu ảnh và mask
                        utils.save_image(
                            all_images,
                            str(self.results_folder / f'sample-{milestone}.png'),
                            nrow=int(math.sqrt(self.num_samples))
                        )
                        utils.save_image(
                            all_masks,
                            str(self.results_folder / f'sample-mask-{milestone}.png'),
                            nrow=int(math.sqrt(self.num_samples))
                        )

                        # FID (nếu bật)
                        if self.calculate_fid:
                            fid_score = self.fid_scorer.fid_score()
                            accelerator.print(f'fid_score: {fid_score}')
                            if self.save_best_and_latest_only:
                                if self.best_fid > fid_score:
                                    self.best_fid = fid_score
                                    self.save("best")
                                self.save("latest")
                        else:
                            self.save(milestone)

                pbar.update(1)

        accelerator.print('training complete')

In [ ]:
model = Unet(
    dim=64,
    channels=3,          # Ảnh RGB (3 kênh)
    num_classes=2,       # Mask nhị phân (2 lớp: nền và đối tượng)
    dim_mults=(1, 2, 4, 8),
    flash_attn=False
)

In [ ]:
diffusion = MultiTaskDiffusion(
    model,
    image_size=128,
    num_classes=2,               
    timesteps=1000,
    sampling_timesteps=250,
    objective='pred_x0',
    beta_schedule='sigmoid',
    balancing_factor=0.684,
    auto_normalize=True
)

In [ ]:
dataset = ImageMaskDataset(
    folder='/kaggle/input/datasets/totrang24082004/otu2d-dataset/dataset_split',
    image_size=128,
    num_classes=2
)

img, mask = dataset[0]

print(img.shape)   # (3,128,128)
print(mask.shape)  # (2,128,128)
print(mask.unique())  # tensor([0., 1.])

In [ ]:
trainer = Trainer(
    diffusion_model=diffusion,           
    folder='/kaggle/input/datasets/totrang24082004/otu2d-dataset/dataset_split',
    train_batch_size=16,
    gradient_accumulate_every=1,
    train_lr=2e-5,
    train_num_steps=100000,
    save_and_sample_every=2000,
    num_samples=9,
    results_folder='/kaggle/working/results',
    amp=False,
    mixed_precision_type='fp16',
    num_classes=2,
)

In [ ]:
trainer.train()